In [1]:
from typing import Any
import timeit
import numpy as np
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
import os
# os.environ["NUMBA_DISABLE_JIT"] = "1"

import elastica as ea
from elasticapp import MemoryBlockCosseratRod

# Try to import set_num_threads if threading is enabled
try:
    from elasticapp._memory_block import set_num_threads, get_max_threads
    THREADING_AVAILABLE = True
except ImportError:
    THREADING_AVAILABLE = False
    raise RuntimeError("Threading not available (ELASTICAPP_USE_THREADING not enabled)")


In [2]:
# Create 50000 rods with 6 elements each, iterate 100 operations
N = 100
n_rods = 50000
n_elems_per_rod = 6

# Create rods
rods = [
    ea.CosseratRod.straight_rod(
        n_elements=n_elems_per_rod,
        start=np.zeros(3),
        direction=np.array([0.0, 0.0, 1.0]),
        normal=np.array([1.0, 0.0, 0.0]),
        base_length=1.0,
        base_radius=0.01,
        density=3000,
        youngs_modulus=1e6,
    )
    for _ in tqdm(range(n_rods))
]

def jitter_rod(rod):
    rng = np.random.default_rng(42)
    rod.external_forces[:] = rng.uniform(-1e-6, 1e-6, rod.external_forces.shape)
    rod.external_torques[:] = rng.uniform(-1e-6, 1e-6, rod.external_torques.shape)

100%|██████████| 50000/50000 [00:24<00:00, 2016.30it/s]


In [3]:
block_rod = ea.MemoryBlockCosseratRod(rods, range(n_rods))
jitter_rod(block_rod)

py_result = {}

for _ in range(10): # Pre-run
    block_rod.compute_internal_forces_and_torques(0.0)
T = timeit.timeit(lambda: block_rod.compute_internal_forces_and_torques(0.0), number=N)
py_result['compute_internal_forces_and_torques'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (compute_internal_forces_and_torques - inverse rotation, equations)")
for _ in range(10): # Pre-run
    block_rod.update_accelerations(0.0)
T = timeit.timeit(lambda: block_rod.update_accelerations(0.0), number=N)
py_result['update_accelerations'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (update_accelerations - block addition)")
for _ in range(10): # Pre-run
    block_rod.update_kinematics(0.0, 1e-4)
T = timeit.timeit(lambda: block_rod.update_kinematics(0.0, 1e-4), number=N)
py_result['update_kinematics'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (update_kinematics - rodrigues)")
for _ in range(10): # Pre-run
    block_rod.update_dynamics(0.0, 1e-4)
T = timeit.timeit(lambda: block_rod.update_dynamics(0.0, 1e-4), number=N)
py_result['update_dynamics'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (update_dynamics)")

Time per call: 42.549 ms (compute_internal_forces_and_torques - inverse rotation, equations)
Time per call: 5.615 ms (update_accelerations - block addition)
Time per call: 15.882 ms (update_kinematics - rodrigues)
Time per call: 0.974 ms (update_dynamics)


In [4]:
assert THREADING_AVAILABLE


# Benchmark parameters
# thread_counts = [1, 2, 4, 8, 16]

# Storage for results
cpp_results_for_threads = []

# Run benchmark for each thread count
# for num_threads in thread_counts:
    # set_num_threads(num_threads)
    # print(f"Threading available: {num_threads}")

rods = [
    ea.CosseratRod.straight_rod(
        n_elements=n_elems_per_rod,
        start=np.zeros(3),
        direction=np.array([0.0, 0.0, 1.0]),
        normal=np.array([1.0, 0.0, 0.0]),
        base_length=1.0,
        base_radius=0.01,
        density=3000,
        youngs_modulus=1e6,
    )
    for _ in tqdm(range(n_rods))
]

block_rod = MemoryBlockCosseratRod(rods, range(n_rods))
jitter_rod(block_rod)

cpp_result = {}
cpp_results_for_threads.append(cpp_result)

for _ in range(10): # Pre-run
    block_rod.compute_internal_forces_and_torques(0.0)
T = timeit.timeit(lambda: block_rod.compute_internal_forces_and_torques(0.0), number=N)
cpp_result['compute_internal_forces_and_torques'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (compute_internal_forces_and_torques - inverse rotation, equations)")
for _ in range(10): # Pre-run
    block_rod.update_accelerations(0.0)
T = timeit.timeit(lambda: block_rod.update_accelerations(0.0), number=N)
cpp_result['update_accelerations'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (update_accelerations - block addition)")
for _ in range(10): # Pre-run
    block_rod.update_kinematics(0.0, 1e-4)
T = timeit.timeit(lambda: block_rod.update_kinematics(0.0, 1e-4), number=N)
cpp_result['update_kinematics'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (update_kinematics - rodrigues)")
for _ in range(10): # Pre-run
    block_rod.update_dynamics(0.0, 1e-4)
T = timeit.timeit(lambda: block_rod.update_dynamics(0.0, 1e-4), number=N)
cpp_result['update_dynamics'] = T/N*1000
print(f"Time per call: {T/N*1000:.3f} ms (update_dynamics)")

print("-"*60)

100%|██████████| 50000/50000 [00:24<00:00, 2057.62it/s]


Time per call: 21.076 ms (compute_internal_forces_and_torques - inverse rotation, equations)
Time per call: 3.041 ms (update_accelerations - block addition)
Time per call: 1.601 ms (update_kinematics - rodrigues)
Time per call: 0.756 ms (update_dynamics)
------------------------------------------------------------
